# QuickDraw Sketch Recognizer — Training Notebook

**Staged experiments** — CNN architecture trained on the Quick, Draw! dataset subset.

## Pipeline
1. Load  files from  dataset directory
2. Pre-process (normalize, reshape, 80/10/10 split)
3. Train baseline CNN (**Experiment 1**)
4. Add dropout + batch-norm (**Experiment 2**)
5. Evaluate on test set — confusion matrix, per-class accuracy
6. Export best model to 


In [ ]:
import os, glob, json, time, warnings
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from datetime import datetime
from tqdm.notebook import tqdm

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, random_split
import torchvision.transforms as T

from sklearn.metrics import classification_report, confusion_matrix

warnings.filterwarnings("ignore")
torch.manual_seed(42)
np.random.seed(42)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {DEVICE}")


In [ ]:
# Locate .npy files — adjust DATA_DIR if needed
DATA_DIR = Path("data/quickdraw")
npy_files = sorted(DATA_DIR.glob("*.npy"))
CLASS_NAMES = [f.stem for f in npy_files]
print(f"Found {len(npy_files)} classes: {CLASS_NAMES}")


In [ ]:
# ─── Load and subsample dataset ─────────────────────────────────────────
MAX_PER_CLASS = 10_000   # samples per class (tune for speed vs accuracy)

all_images, all_labels = [], []
for idx, npy_path in enumerate(npy_files):
    data = np.load(npy_path)  # shape: (N, 784)
    n = min(MAX_PER_CLASS, len(data))
    sampled = data[:n]
    all_images.append(sampled)
    all_labels.extend([idx] * n)
    print(f"  [{idx+1}/{len(npy_files)}] {npy_path.stem}: {n:,} samples")

X = np.concatenate(all_images, axis=0).astype(np.float32) / 255.0
y = np.array(all_labels, dtype=np.int64)
print(f"
Total dataset: {len(X):,} samples, {len(CLASS_NAMES)} classes")


In [ ]:
class QuickDrawDataset(Dataset):
    def __init__(self, images: np.ndarray, labels: np.ndarray, augment: bool = False):
        self.images = images  # (N, 784) float32, [0,1]
        self.labels = labels
        self.augment = augment
        self.to_tensor = T.Compose([
            T.ToTensor(),
        ])
        if augment:
            self.aug = T.Compose([
                T.RandomHorizontalFlip(p=0.3),
                T.RandomRotation(degrees=15),
                T.RandomAffine(degrees=0, translate=(0.1, 0.1)),
            ])

    def __len__(self):
        return len(self.images)

    def __getitem__(self, idx):
        # Reshape to (28, 28), invert so strokes are black on white becomes white-on-black
        img = self.images[idx].reshape(28, 28)
        img = 1.0 - img   # invert polarity to match training convention
        img_pil = __import__("PIL").Image.fromarray((img * 255).astype(np.uint8), mode="L")
        if self.augment:
            img_pil = self.aug(img_pil)
        tensor = self.to_tensor(img_pil)  # shape: (1, 28, 28)
        return tensor, self.labels[idx]

# 80 / 10 / 10 split
n_total = len(X)
n_train = int(0.8 * n_total)
n_val = int(0.1 * n_total)
n_test = n_total - n_train - n_val

full_ds = QuickDrawDataset(X, y)
train_ds, val_ds, test_ds = random_split(full_ds, [n_train, n_val, n_test], generator=torch.Generator().manual_seed(42))

# Enable augmentation only on train set
train_ds.dataset.augment = True

BATCH = 256
train_loader = DataLoader(train_ds, batch_size=BATCH, shuffle=True, num_workers=0, pin_memory=DEVICE.type=="cuda")
val_loader   = DataLoader(val_ds,   batch_size=BATCH, shuffle=False, num_workers=0)
test_loader  = DataLoader(test_ds,  batch_size=BATCH, shuffle=False, num_workers=0)

print(f"Train: {n_train:,}  Val: {n_val:,}  Test: {n_test:,}")


In [ ]:
class SketchCNN(nn.Module):
    """
    SketchCNN — lightweight 3-block CNN for 28x28 grayscale Quick Draw! sketches.
    Exp 1: No dropout / BN.  Exp 2: adds BN + dropout for regularization.
    """
    def __init__(self, num_classes: int, use_bn: bool = True, dropout: float = 0.4):
        super().__init__()
        self.features = nn.Sequential(
            # Block 1
            nn.Conv2d(1, 32, 3, padding=1),
            nn.BatchNorm2d(32) if use_bn else nn.Identity(),
            nn.ReLU(inplace=True),
            nn.Conv2d(32, 32, 3, padding=1),
            nn.BatchNorm2d(32) if use_bn else nn.Identity(),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2),           # -> 14x14
            nn.Dropout2d(dropout/2) if use_bn else nn.Identity(),

            # Block 2
            nn.Conv2d(32, 64, 3, padding=1),
            nn.BatchNorm2d(64) if use_bn else nn.Identity(),
            nn.ReLU(inplace=True),
            nn.Conv2d(64, 64, 3, padding=1),
            nn.BatchNorm2d(64) if use_bn else nn.Identity(),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2),           # -> 7x7
            nn.Dropout2d(dropout/2) if use_bn else nn.Identity(),

            # Block 3
            nn.Conv2d(64, 128, 3, padding=1),
            nn.BatchNorm2d(128) if use_bn else nn.Identity(),
            nn.ReLU(inplace=True),
            nn.AdaptiveAvgPool2d((3, 3)),  # -> 3x3
        )
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(128 * 3 * 3, 512),
            nn.ReLU(inplace=True),
            nn.Dropout(dropout) if use_bn else nn.Identity(),
            nn.Linear(512, num_classes),
        )

    def forward(self, x):
        return self.classifier(self.features(x))

print(f"Model params: {sum(p.numel() for p in SketchCNN(len(CLASS_NAMES)).parameters()):,}")


In [ ]:
def train_experiment(model, name: str, epochs: int = 15, lr: float = 1e-3):
    model = model.to(DEVICE)
    optimizer = optim.Adam(model.parameters(), lr=lr, weight_decay=1e-4)
    scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs)
    criterion = nn.CrossEntropyLoss()

    history = {"train_loss": [], "val_loss": [], "train_acc": [], "val_acc": []}
    best_val_acc = 0.0
    best_state = None

    for epoch in range(1, epochs + 1):
        # ── Train ────────────────────────────────────
        model.train()
        t_loss, t_correct, t_total = 0.0, 0, 0
        for xb, yb in tqdm(train_loader, desc=f"[{name}] E{epoch}/{epochs}", leave=False):
            xb, yb = xb.to(DEVICE), yb.to(DEVICE)
            optimizer.zero_grad()
            out = model(xb)
            loss = criterion(out, yb)
            loss.backward()
            optimizer.step()
            t_loss += loss.item() * len(xb)
            t_correct += (out.argmax(1) == yb).sum().item()
            t_total += len(xb)
        scheduler.step()

        # ── Validate ─────────────────────────────────
        model.eval()
        v_loss, v_correct, v_total = 0.0, 0, 0
        with torch.no_grad():
            for xb, yb in val_loader:
                xb, yb = xb.to(DEVICE), yb.to(DEVICE)
                out = model(xb)
                v_loss += criterion(out, yb).item() * len(xb)
                v_correct += (out.argmax(1) == yb).sum().item()
                v_total += len(xb)

        ta = t_correct / t_total
        va = v_correct / v_total
        history["train_loss"].append(t_loss / t_total)
        history["val_loss"].append(v_loss / v_total)
        history["train_acc"].append(ta)
        history["val_acc"].append(va)

        print(f"  E{epoch:02d} | train_loss={t_loss/t_total:.4f} acc={ta:.3f} | val_loss={v_loss/v_total:.4f} acc={va:.3f}")

        if va > best_val_acc:
            best_val_acc = va
            best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}

    model.load_state_dict(best_state)
    print(f"
[{name}] Best val acc: {best_val_acc:.4f}")
    return model, history, best_val_acc


In [ ]:
print("=" * 60)
print("Experiment 1 — Baseline CNN (no BN, no dropout)")
print("=" * 60)

model_exp1 = SketchCNN(len(CLASS_NAMES), use_bn=False, dropout=0.0)
model_exp1, hist1, acc1 = train_experiment(model_exp1, "Exp1", epochs=10)


In [ ]:
print("=" * 60)
print("Experiment 2 — CNN + BatchNorm + Dropout")
print("=" * 60)

model_exp2 = SketchCNN(len(CLASS_NAMES), use_bn=True, dropout=0.4)
model_exp2, hist2, acc2 = train_experiment(model_exp2, "Exp2", epochs=15)

# Choose best model
best_model = model_exp2 if acc2 >= acc1 else model_exp1
print(f"
Best model: Exp2={acc2:.4f}  Exp1={acc1:.4f}")


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ax, hist, name in [(axes[0], hist1, "Exp1 (Baseline)"), (axes[1], hist2, "Exp2 (BN+Dropout)")]:
    epochs_range = range(1, len(hist["train_acc"]) + 1)
    ax.plot(epochs_range, hist["train_acc"], label="Train Acc", marker="o", markersize=3)
    ax.plot(epochs_range, hist["val_acc"],   label="Val Acc",   marker="s", markersize=3)
    ax.set_title(name)
    ax.set_xlabel("Epoch")
    ax.set_ylabel("Accuracy")
    ax.legend()
    ax.grid(alpha=0.3)

plt.tight_layout()
plt.savefig("ml/sketch_recognizer/artifacts/training_curves.png", dpi=120)
plt.show()


In [ ]:
best_model.eval()
all_preds, all_true = [], []

with torch.no_grad():
    for xb, yb in test_loader:
        out = best_model(xb.to(DEVICE))
        all_preds.extend(out.argmax(1).cpu().numpy())
        all_true.extend(yb.numpy())

print(classification_report(all_true, all_preds, target_names=CLASS_NAMES, digits=4))

# Confusion matrix
cm = confusion_matrix(all_true, all_preds)
plt.figure(figsize=(max(8, len(CLASS_NAMES)), max(6, len(CLASS_NAMES))))
sns.heatmap(cm, annot=True, fmt="d", xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES, cmap="Blues")
plt.title("Confusion Matrix — Test Set")
plt.ylabel("True")
plt.xlabel("Predicted")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.savefig("ml/sketch_recognizer/artifacts/confusion_matrix.png", dpi=120)
plt.show()


In [ ]:
ARTIFACTS_DIR = Path("ml/sketch_recognizer/artifacts")
ARTIFACTS_DIR.mkdir(parents=True, exist_ok=True)

model_path = ARTIFACTS_DIR / "best_model.pt"
torch.save({
    "model_state_dict": best_model.state_dict(),
    "class_names": CLASS_NAMES,
    "num_classes": len(CLASS_NAMES),
    "architecture": "SketchCNN",
    "use_bn": True,
    "dropout": 0.4,
    "trained_at": datetime.utcnow().isoformat(),
    "val_accuracy_exp1": acc1,
    "val_accuracy_exp2": acc2,
}, model_path)

print(f"Model saved to {model_path}  ({model_path.stat().st_size / 1e6:.1f} MB)")

# Also write a metadata JSON for the FastAPI service
meta = {
    "class_names": CLASS_NAMES,
    "num_classes": len(CLASS_NAMES),
    "model_file": str(model_path.resolve()),
}
(ARTIFACTS_DIR / "model_meta.json").write_text(json.dumps(meta, indent=2))
print("Metadata JSON written.")
